In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os
import gc
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import tifffile as tiff

import warnings

warnings.filterwarnings("ignore")

try:
    import segmentation_models_pytorch as smp  # type: ignore

    _HAS_SMP = True
except Exception:
    smp = None
    _HAS_SMP = False

try:
    import pyvips  # type: ignore

    _HAS_PYVIPS = True
except Exception:
    pyvips = None
    _HAS_PYVIPS = False

try:
    import openslide  # type: ignore

    _HAS_OPENSLIDE = True
except Exception:
    openslide = None
    _HAS_OPENSLIDE = False



In [2]:
sz = 256  # the size of tiles
reduce = 4  # reduce the original images by 4 times
TH = 0.44  # threshold for positive predictions

DATA = "../input/hubmap-kidney-segmentation/test/"
if not os.path.isdir(DATA):
    alt = "/kaggle/input/hubmap-kidney-segmentation/test/"
    if os.path.isdir(alt):
        DATA = alt

MODELS = [
    f"../input/normalize-seresnext/result/se_resnext50_32x4d-FOLD-{i}-model.pth"
    for i in range(5)
]

df_sample = pd.read_csv("../input/hubmap-kidney-segmentation/sample_submission.csv")
if not set(["id", "predicted"]).issubset(df_sample.columns):
    df_sample.columns = ["id", "predicted"]

bs = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [3]:
def enc2mask(encs, shape):
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for m, enc in enumerate(encs):
        if isinstance(enc, (float, np.floating)) and np.isnan(enc):
            continue
        s = enc.split()
        for i in range(len(s) // 2):
            start = int(s[2 * i]) - 1
            length = int(s[2 * i + 1])
            img[start : start + length] = 1 + m
    return img.reshape(shape).T


def mask2enc(mask, n=1):
    pixels = mask.T.flatten()
    encs = []
    for i in range(1, n + 1):
        p = (pixels == i).astype(np.int8)
        if p.sum() == 0:
            encs.append(np.nan)
        else:
            p = np.concatenate([[0], p, [0]])
            runs = np.where(p[1:] != p[:-1])[0] + 1
            runs[1::2] -= runs[::2]
            encs.append(" ".join(str(x) for x in runs))
    return encs


def rle_encode_less_memory(img):
    pixels = img.T.flatten()
    if pixels.size == 0:
        return ""
    pixels[0] = 0
    pixels[-1] = 0
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 2
    if runs.size == 0:
        return ""
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)




In [4]:
mean = np.array([0.65459856, 0.48386562, 0.69428385])
std = np.array([0.15167958, 0.23584107, 0.13146145])

s_th = 40  # saturation blanking threshold
p_th = 1000 * (sz // 256) ** 2  # threshold for the minimum number of pixels


def img2tensor(img, dtype: np.dtype = np.float32):
    if img.ndim == 2:
        img = np.expand_dims(img, 2)
    img = np.transpose(img, (2, 0, 1))
    return torch.from_numpy(img.astype(dtype, copy=False))


def _tile_to_rgb_uint8(tile: np.ndarray) -> np.ndarray:
    """Convert (H,W), (H,W,C), or (C,H,W) tile to HWC RGB uint8."""
    if tile.ndim == 2:
        tile = np.stack([tile, tile, tile], axis=-1)
    elif tile.ndim == 3:
        if tile.shape[-1] in (3, 4):  # HWC
            tile = tile[..., :3]
        elif tile.shape[0] in (3, 4):  # CHW
            tile = tile[:3, ...]
            tile = np.transpose(tile, (1, 2, 0))
        else:
            tile = tile[..., :3]
    else:
        raise ValueError(f"Unsupported tile ndim={tile.ndim}")

    if tile.dtype == np.uint16:
        tile = (tile / 257.0).astype(np.uint8)
    elif tile.dtype != np.uint8:
        tile = np.clip(tile, 0, 255).astype(np.uint8)
    return tile


class _TiffTileSource:
    """
    Score-impacting fix (I/O only): make OpenSlide region reads align with the target
    reduced-space tile coordinates by reading from the best pyramid level and mapping
    coordinates in *that level*, not by guessing level-0 then resizing.
    This preserves tiling/model/threshold/RLE semantics but prevents blank/incorrect tiles
    that lead to empty predictions (0.0 Dice).
    """

    def __init__(self, path: str, reduce_factor: int):
        self.path = path
        self.reduce_factor = int(reduce_factor)

        with tiff.TiffFile(self.path) as tf:
            page = tf.pages[0]
            shape = page.shape
            if len(shape) == 2:
                h, w = shape
            elif len(shape) == 3:
                if shape[-1] in (3, 4):
                    h, w = shape[0], shape[1]
                elif shape[0] in (3, 4):
                    h, w = shape[1], shape[2]
                else:
                    h, w = shape[0], shape[1]
            else:
                raise ValueError(f"Unexpected TIFF shape {shape} for {path}")

        self.full_h = int(h)
        self.full_w = int(w)
        self.h = int(np.ceil(self.full_h / self.reduce_factor))
        self.w = int(np.ceil(self.full_w / self.reduce_factor))

        self._osr = None
        self._os_level = 0
        self._os_downsample = 1.0
        if _HAS_OPENSLIDE:
            try:
                self._osr = openslide.OpenSlide(self.path)
                downs = np.array(self._osr.level_downsamples, dtype=np.float64)
                if downs.size > 0:
                    # pick closest level to our reduce factor
                    self._os_level = int(
                        np.argmin(np.abs(downs - float(self.reduce_factor)))
                    )
                    self._os_downsample = float(
                        self._osr.level_downsamples[self._os_level]
                    )
            except Exception:
                self._osr = None

        self._mm = None
        try:
            mm = tiff.memmap(self.path)
            if mm.ndim == 2:
                mm = mm[..., None]
            elif mm.ndim == 3 and mm.shape[0] in (3, 4) and mm.shape[-1] not in (3, 4):
                mm = np.transpose(mm, (1, 2, 0))
            self._mm = mm
        except Exception:
            self._mm = None

        self._vips = None
        if _HAS_PYVIPS:
            try:
                self._vips = pyvips.Image.new_from_file(self.path, access="sequential")
            except Exception:
                self._vips = None

    def _read_via_openslide(self, p00: int, p01: int, p10: int, p11: int) -> np.ndarray:
        out_h = max(0, int(p01 - p00))
        out_w = max(0, int(p11 - p10))
        if out_h == 0 or out_w == 0:
            return np.zeros((0, 0, 3), dtype=np.uint8)

        level = int(self._os_level)
        ds = float(self._os_downsample)

        # --- Change: map reduced-space coords -> level coords consistently.
        # We want coordinates in "reduced space" where 1 px ~ reduce_factor px at level0.
        # At OpenSlide level with downsample ds, 1 level-px ~ ds level0-px.
        # So reduced->level scale is (reduce_factor / ds).
        scale = float(self.reduce_factor) / ds

        x_level = int(np.floor(p10 * scale))  # col -> x
        y_level = int(np.floor(p00 * scale))  # row -> y
        w_level = max(1, int(np.ceil(out_w * scale)))
        h_level = max(1, int(np.ceil(out_h * scale)))

        region = self._osr.read_region((x_level, y_level), level, (w_level, h_level))
        arr = np.array(region, dtype=np.uint8)[..., :3]

        # Keep semantics: always return exactly (out_h, out_w) via area resize if needed.
        if arr.shape[0] != out_h or arr.shape[1] != out_w:
            arr = cv2.resize(arr, (out_w, out_h), interpolation=cv2.INTER_AREA)

        return _tile_to_rgb_uint8(arr)

    def _read_via_memmap(self, p00: int, p01: int, p10: int, p11: int) -> np.ndarray:
        r = self.reduce_factor
        x0, x1 = int(p00 * r), int(p01 * r)
        y0, y1 = int(p10 * r), int(p11 * r)
        tile = self._mm[x0:x1:r, y0:y1:r]
        return _tile_to_rgb_uint8(tile)

    def _read_via_pyvips(self, p00: int, p01: int, p10: int, p11: int) -> np.ndarray:
        r = self.reduce_factor
        x0, x1 = int(p00 * r), int(p01 * r)
        y0, y1 = int(p10 * r), int(p11 * r)
        width = max(0, y1 - y0)
        height = max(0, x1 - x0)
        if width == 0 or height == 0:
            return np.zeros((0, 0, 3), dtype=np.uint8)

        crop = self._vips.crop(y0, x0, width, height)

        if crop.bands == 1:
            crop = crop.bandjoin([crop, crop])
        elif crop.bands >= 4:
            crop = crop.extract_band(0, n=3)

        fmt = crop.format
        if fmt not in ("uchar", "ushort"):
            crop = crop.cast("uchar")
        mem = crop.write_to_memory()
        arr = np.frombuffer(
            mem, dtype=np.uint8 if crop.format == "uchar" else np.uint16
        )
        arr = arr.reshape(crop.height, crop.width, crop.bands)

        arr = arr[::r, ::r]
        return _tile_to_rgb_uint8(arr)

    def read(self, p00: int, p01: int, p10: int, p11: int) -> np.ndarray:
        p00 = max(0, min(int(p00), self.h))
        p01 = max(0, min(int(p01), self.h))
        p10 = max(0, min(int(p10), self.w))
        p11 = max(0, min(int(p11), self.w))

        if p01 <= p00 or p11 <= p10:
            return np.zeros((0, 0, 3), dtype=np.uint8)

        if self._osr is not None:
            try:
                return self._read_via_openslide(p00, p01, p10, p11)
            except Exception:
                pass

        if self._mm is not None:
            try:
                return self._read_via_memmap(p00, p01, p10, p11)
            except Exception:
                pass

        if self._vips is not None:
            try:
                return self._read_via_pyvips(p00, p01, p10, p11)
            except Exception:
                pass

        return np.zeros((p01 - p00, p11 - p10, 3), dtype=np.uint8)


class HuBMAPDataset(Dataset):
    def __init__(self, idx, sz=sz, reduce=reduce):
        self.idx = idx
        self.path = os.path.join(DATA, idx + ".tiff")

        self.reduce = int(reduce)
        self.sz = int(sz)  # tile size in reduced space

        self._src = _TiffTileSource(self.path, reduce_factor=self.reduce)

        self.shape = (self._src.h, self._src.w)

        self.pad0 = (self.sz - self.shape[0] % self.sz) % self.sz
        self.pad1 = (self.sz - self.shape[1] % self.sz) % self.sz
        self.n0max = (self.shape[0] + self.pad0) // self.sz
        self.n1max = (self.shape[1] + self.pad1) // self.sz

    def __len__(self):
        return self.n0max * self.n1max

    def __getitem__(self, idx):
        n0, n1 = idx // self.n1max, idx % self.n1max
        x0, y0 = -self.pad0 // 2 + n0 * self.sz, -self.pad1 // 2 + n1 * self.sz
        p00, p01 = max(0, x0), min(x0 + self.sz, self.shape[0])
        p10, p11 = max(0, y0), min(y0 + self.sz, self.shape[1])

        tile = np.zeros((self.sz, self.sz, 3), np.uint8)
        patch = self._src.read(p00, p01, p10, p11)
        tile[(p00 - x0) : (p01 - x0), (p10 - y0) : (p11 - y0)] = patch

        hsv = cv2.cvtColor(tile, cv2.COLOR_RGB2HSV)
        h, s, v = cv2.split(hsv)

        if (s > s_th).sum() <= p_th or tile.sum() <= p_th:
            return img2tensor((tile / 255.0 - mean) / std), -1
        else:
            return img2tensor((tile / 255.0 - mean) / std), idx




In [5]:
class Model_pred:
    def __init__(self, models, dl, tta: bool = False, half: bool = False):
        self.models = models
        self.dl = dl
        self.tta = tta
        self.half = half

    def __iter__(self):
        with torch.no_grad():
            for x, y in iter(self.dl):
                if (y >= 0).sum() > 0:  # exclude empty tiles
                    x = x[y >= 0].to(device)
                    y = y[y >= 0]
                    if self.half:
                        x = x.half()

                    py = None
                    for model in self.models:
                        p = model(x)
                        p = torch.sigmoid(p).detach()
                        py = p if py is None else (py + p)

                    if self.tta:
                        flips = [[-1], [-2], [-2, -1]]
                        for f in flips:
                            xf = torch.flip(x, f)
                            for model in self.models:
                                p = model(xf)
                                p = torch.flip(p, f)
                                py += torch.sigmoid(p).detach()
                        py /= 1 + len(flips)

                    py /= len(self.models)

                    py = py.permute(0, 2, 3, 1).float().cpu()

                    batch_size = len(py)
                    for i in range(batch_size):
                        yield py[i], y[i]

    def __len__(self):
        return len(self.dl.dataset)




In [6]:
class _ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class _TinyUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.enc1 = _ConvBlock(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = _ConvBlock(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = _ConvBlock(base * 2, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = _ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = _ConvBlock(base * 2, base)

        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        d2 = self.up2(e3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.out(d1)


class HuBMAP(nn.Module):
    def __init__(self):
        super(HuBMAP, self).__init__()
        if _HAS_SMP:
            self.cnn_model = smp.Unet(
                "se_resnext50_32x4d", encoder_weights=None, classes=1
            )
        else:
            self.cnn_model = _TinyUNet(in_ch=3, out_ch=1, base=32)

    def forward(self, imgs):
        img_segs = self.cnn_model(imgs)
        return img_segs




In [7]:
models = []
missing = []
for path in MODELS:
    if os.path.isfile(path):
        state_dict = torch.load(path, map_location=torch.device("cpu"))
        model = HuBMAP()
        model.load_state_dict(state_dict)
        model.float()
        model.eval()
        model.to(device)
        models.append(model)
        del state_dict
    else:
        missing.append(path)

if len(models) == 0:
    model = HuBMAP()
    model.float()
    model.eval()
    model.to(device)
    models = [model]

gc.collect()



100

In [8]:
names, preds = [], []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    idx = row["id"]
    ds = HuBMAPDataset(idx)
    dl = DataLoader(ds, batch_size=bs, pin_memory=True, shuffle=False, num_workers=0)
    mp = Model_pred(models, dl)

    mask = torch.zeros(len(ds), ds.sz, ds.sz, dtype=torch.int8)

    nonempty_tiles = 0
    for p, i in iter(mp):
        nonempty_tiles += 1
        mask[i.item()] = p.squeeze(-1) > TH

    if nonempty_tiles == 0:
        print(f"Warning: {idx} produced 0 non-empty tiles. Check TIFF reader backends.")

    mask = (
        mask.view(ds.n0max, ds.n1max, ds.sz, ds.sz)
        .permute(0, 2, 1, 3)
        .reshape(ds.n0max * ds.sz, ds.n1max * ds.sz)
    )
    mask = mask[
        ds.pad0 // 2 : -(ds.pad0 - ds.pad0 // 2) if ds.pad0 > 0 else ds.n0max * ds.sz,
        ds.pad1 // 2 : -(ds.pad1 - ds.pad1 // 2) if ds.pad1 > 0 else ds.n1max * ds.sz,
    ]

    rle = rle_encode_less_memory(mask.numpy())
    names.append(idx)
    preds.append(rle)

    del mask, ds, dl
    gc.collect()



  0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
df = pd.DataFrame({"id": names, "predicted": preds})
df = df_sample[["id"]].merge(df, on="id", how="left")
df["predicted"] = df["predicted"].fillna("")  # empty mask allowed

df.to_csv("submission.csv", index=False)
print(df.head())
print("Wrote submission.csv with rows:", len(df))
if len(missing) > 0:
    print(
        f"Warning: {len(missing)} model weight files were missing; ran with {len(models)} model(s) available."
    )
if not _HAS_OPENSLIDE:
    print(
        "Warning: openslide not available; JPEG-compressed TIFF region reading may fail, leading to empty predictions."
    )
elif not _HAS_PYVIPS and not _HAS_OPENSLIDE:
    print(
        "Warning: neither openslide nor pyvips available; JPEG-compressed TIFF region reading may fail, leading to empty predictions."
    )

          id                                          predicted
0  8242609fa  5047839 3584 5055664 3584 5063489 3584 5071314...
1  0486052bb  5880952 768 5887398 768 5893844 768 5900290 76...
2  095bf7a1f                                                   
Wrote submission.csv with rows: 3
